<center><ins><h1>Data Statistics</h1></ins></center>

###### **Author:** Cedric Hering-Peter
###### **Address:** AG Schulz / Botantical Institut / Am Botanischen Garten 5 / 24118 Kiel

# Imports

In [6]:

# System
import os, sys

WORKSPACE = os.path.abspath(os.path.join(os.getcwd(), ".."))
if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)

# Packages
import pandas as pd
import numpy as np
from scripts import data_helper, plot_helper
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.weightstats import ttest_ind
#import pylab

# Further statistics
import statsmodels.stats.api as sms
import scikit_posthocs as sp

CONFIG = data_helper.load_config()
META_DATA = data_helper.load_meta_data(CONFIG)
plot_helper.set_config(CONFIG)

# Variables

In [7]:

# Show numeric output in decimal format e.g., 2.1514
pd.options.display.float_format = '{:,.2f}'.format

# Raw Data Import

In [8]:
# Get all raw data files
rec_rat = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Recovery-Rates_Raw-Data.csv"))
sin_vel = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Sinking-Velocities_Raw-Data.csv"))
par_siz = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Particle-Sizes_Raw-Data.csv"))
sus_sol = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Suspended-Solids_Raw-Data.csv"))
zet_pot = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Zeta-Potential_Raw-Data.csv"))

# Average techniquel replicates of zeta potentials
zet_tec_rep_mean = zet_pot.groupby(["ID", "EXPERIMENT_NAME", "SAMPLE_NAME", "TIME", "BIO_REP"], as_index = False).agg({"ZETA_POTENTIAL_MV": pd.Series.mean})
zet_tec_rep_mean.drop(zet_tec_rep_mean[zet_tec_rep_mean["EXPERIMENT_NAME"].str.contains("Control")].index, inplace = True)
zet_tec_rep_mean.drop(zet_tec_rep_mean[zet_tec_rep_mean["SAMPLE_NAME"] == "+EPS"].index, inplace = True)
zet_pot = zet_tec_rep_mean.sort_values(by = ["EXPERIMENT_NAME", "TIME", "SAMPLE_NAME", "BIO_REP"], ignore_index = True)

sus_sol = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Suspended-Solids_Raw-Data.csv"))
sus_sol.drop(sus_sol[sus_sol["SAMPLE_NAME"] == "+EPS"].index, inplace = True)
sus_sol = sus_sol.sort_values(by = ["EXPERIMENT_NAME", "TIME", "SAMPLE_NAME", "BIO_REP"], ignore_index = True)


# Merge into dataframe with all categories
merged_df = pd.concat([rec_rat["ID"], rec_rat["EXPERIMENT_NAME"], rec_rat["SAMPLE_NAME"], rec_rat["TIME"], 
                       rec_rat["RRATE_%"], 
                       sin_vel["MEAN_SVELO_M/H"], 
                       par_siz["GEOMETRIC.MEAN"],
                       sus_sol["TSS_g/L"],
                       sus_sol["TSS_%"],
                       sus_sol["VSS_g/L"],
                       sus_sol["VSS_%"],
                       zet_pot["ZETA_POTENTIAL_MV"],
                       ], axis = 1)
merged_df.rename(columns = {"GEOMETRIC.MEAN": "MEAN_PARSIZE_µM"}, inplace = True)
merged_df.head()

,ID,EXPERIMENT_NAME,SAMPLE_NAME,TIME,RRATE_%,MEAN_SVELO_M/H,MEAN_PARSIZE_µM,TSS_g/L,TSS_%,VSS_g/L,VSS_%,ZETA_POTENTIAL_MV
0,CH230130,SC,A.obliquus,0.00,2.72,0.00,632.67,1.29,95.23,0.06,4.77,-18.23
1,CH230130,SC,A.obliquus,0.00,0.34,0.00,981.62,1.10,94.36,0.06,5.64,-18.20
2,CH230130,SC,A.obliquus,0.00,2.37,0.00,728.29,1.29,95.18,0.06,4.82,-16.50
3,CH230130,SC,Algaemix,0.00,94.85,3.99,632.67,1.29,95.23,0.06,4.77,-18.40
4,CH230130,SC,Algaemix,0.00,98.46,4.75,981.62,1.10,94.36,0.06,5.64,-29.93


# One-Way ANOVA Special cases

In [13]:
owanova_df = pd.DataFrame(data_helper.filter(merged_df, time = [12]))

# One-Way ANOVA over Species Experiment
sc_exp = owanova_df[owanova_df["EXPERIMENT_NAME"] == "SC"]
for index in range(4, len(sc_exp.columns)):
    # Subset data and rename current method to TREATMENT for ANOVA structure
    cur_set = sc_exp.iloc[:, [0,1,2,3, index]]
    method = cur_set.columns[4] # save current method name
    cur_set.columns = ["ID", "EXPERIMENT_NAME", "SAMPLE_NAME", "TIME", "TREATMENT"] # rename treatment column
    
    # Perform one-way ANOVA
    model = ols('TREATMENT ~ C(SAMPLE_NAME)', data = cur_set).fit()
    sc_result = sm.stats.anova_lm(model, typ = 1)
    
    sc_result.index = [sc_exp.columns[index], "Residual"]
    sc_result.index.names = ["SC Experiment"]
    sc_result["F"] = sc_result["F"].apply(lambda x: f"{x:.2f}")
        
    # Save results in csv
    file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Data-Statistics/One-Way-ANOVA-Data.csv")
    sc_result.to_csv(file_name, encoding = "utf-8", index = True, header = True, mode = "a")    
    
# One-Way ANOVA over Species Experiment
cc_exp = owanova_df[owanova_df["EXPERIMENT_NAME"] == "CC"]
for index in range(4, len(cc_exp.columns)):
    # Subset data and rename current method to TREATMENT for ANOVA structure
    cur_set = cc_exp.iloc[:, [0,1,2,3, index]]
    method = cur_set.columns[4] # save current method name
    cur_set.columns = ["ID", "EXPERIMENT_NAME", "SAMPLE_NAME", "TIME", "TREATMENT"] # rename treatment column
    
    # Perform one-way ANOVA
    model = ols('TREATMENT ~ C(SAMPLE_NAME)', data = cur_set).fit()
    cc_result = sm.stats.anova_lm(model, typ = 1)

    cc_result.index = [sc_exp.columns[index], "Residual"]
    cc_result.index.names = ["CC Experiment"]
    cc_result["F"] = cc_result["F"].apply(lambda x: f"{x:.2f}")

    # Save results in csv
    file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Data-Statistics/One-Way-ANOVA-Data.csv")
    cc_result.to_csv(file_name, encoding = "utf-8", index = True, header = True, mode = "a") 

ValueError: negative dimensions are not allowed

# Two-Way ANOVA Physiology Experiments

In [14]:
# Clean all unneeded experiment data 
twanova_df = pd.DataFrame(data_helper.filter(merged_df, time = [12]))
twanova_df.drop(twanova_df[twanova_df["EXPERIMENT_NAME"] == "SC"].index, inplace = True)
twanova_df.drop(twanova_df[twanova_df["EXPERIMENT_NAME"] == "CC"].index, inplace = True)
twanova_df.drop(twanova_df[twanova_df["EXPERIMENT_NAME"] == "RM"].index, inplace = True)
twanova_df.drop(twanova_df[twanova_df["EXPERIMENT_NAME"] == "BM"].index, inplace = True)

# Through all experiments
for exp in twanova_df["EXPERIMENT_NAME"].unique():
    sub_df = twanova_df[twanova_df["EXPERIMENT_NAME"] == exp]
    # For all methods
    for index in range(4, len(sub_df.columns)):
        current_anovaset = sub_df.iloc[:, [0,1,2,3, index]]
        current_exp = current_anovaset["EXPERIMENT_NAME"].iloc[0]
        method = "_".join(current_anovaset.columns[4].split("_")[:-1]) # save current method name
        current_anovaset.columns = ["ID", "EXPERIMENT_NAME", "SAMPLE_NAME", "TIME", "TREATMENT"] # rename treatment column
        
        # Test normal distribution in qqplot
        plot_helper.visualize_qqplot(title = current_exp + " " + method, data = current_anovaset["TREATMENT"])
        # Perform two-way ANOVA
        model = ols('TREATMENT ~ C(SAMPLE_NAME) + C(TIME) + C(SAMPLE_NAME):C(TIME)', data = current_anovaset).fit()
        tw_result = sm.stats.anova_lm(model, typ=2)
        # Insert meta information into result table
        new_col = [method, "Time", "Treatment:Time", "Residual"]
        tw_result.insert(0, column = (current_exp + "-Treatment"), value = new_col)
        tw_result["F"] = tw_result["F"].apply(lambda x: f"{x:.2f}")
        
        # Export to raw data to csv-file
        file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Data-Statistics/Two-Way-ANOVA-Data.csv")
        tw_result.to_csv(file_name, encoding = "utf-8", index = False, header = True, mode = "a")
print("All Physiology_Two-Way-ANOVAs were performed.")

IndexError: single positional indexer is out-of-bounds

# Two-Way ANOVA RM Experiment

In [15]:
# Clean all unneeded experiment data 
twanova_df = pd.DataFrame(data_helper.filter(merged_df, time = [12]))
twanova_df.drop(twanova_df[twanova_df["SAMPLE_NAME"] == "+EPS"].index, inplace = True)

rm_exp = twanova_df[twanova_df["EXPERIMENT_NAME"] == "RM"]
for index in range(4, len(rm_exp.columns)):
    # TW_df MMCvsdH2O
    con_con = pd.DataFrame(rm_exp)
    con_con.drop(con_con[con_con["SAMPLE_NAME"] == "AT"].index, inplace = True)
    con_con.drop(con_con[con_con["SAMPLE_NAME"] == "ET"].index, inplace = True)
    # Subset data and rename current method to TREATMENT for ANOVA structure
    cur_set = con_con.iloc[:, [0,1,2,3, index]]
    current_exp = cur_set["EXPERIMENT_NAME"].iloc[0]
    method = "_".join(cur_set.columns[4].split("_")[:-1]) # save current method name
    cur_set.columns = ["ID", "EXPERIMENT_NAME", "SAMPLE_NAME", "TIME", "TREATMENT"] # rename treatment column
    
    # Test normal distribution in qqplot
    plot_helper.visualize_qqplot(title = current_exp + " " + method, data = cur_set["TREATMENT"])
    # Perform two-way ANOVA
    model = ols('TREATMENT ~ C(SAMPLE_NAME) + C(TIME) + C(SAMPLE_NAME):C(TIME)', data = cur_set).fit()
    tw_rm_result = sm.stats.anova_lm(model, typ=2)
    # Insert meta information into result table
    new_col = [method, "Time", "Treatment:Time", "Residual"]
    tw_rm_result.insert(0, column = (current_exp + "-MMCvsdH2O"), value = new_col)
    tw_rm_result["F"] = tw_rm_result["F"].apply(lambda x: f"{x:.2f}")
    
    # Export to raw data to csv-file
    file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Data-Statistics/Two-Way-ANOVA-Data.csv")
    tw_rm_result.to_csv(file_name, encoding = "utf-8", index = False, header = True, mode = "a")
    
    # TW_df dH2OvsTreatments
    con_treat = pd.DataFrame(rm_exp)
    con_treat.drop(con_treat[con_treat["SAMPLE_NAME"] == "Control"].index, inplace = True)
    # Subset data and rename current method to TREATMENT for ANOVA structure
    cur_set = con_treat.iloc[:, [0,1,2,3, index]]
    current_exp = cur_set["EXPERIMENT_NAME"].iloc[0]
    method = "_".join(cur_set.columns[4].split("_")[:-1]) # save current method name
    cur_set.columns = ["ID", "EXPERIMENT_NAME", "SAMPLE_NAME", "TIME", "TREATMENT"] # rename treatment column
    
    # Test normal distribution in qqplot
    plot_helper.visualize_qqplot(title = current_exp + " " + method, data = cur_set["TREATMENT"])
    # Perform two-way ANOVA
    model = ols('TREATMENT ~ C(SAMPLE_NAME) + C(TIME) + C(SAMPLE_NAME):C(TIME)', data = cur_set).fit()
    tw_rm_result = sm.stats.anova_lm(model, typ=2)
    # Insert meta information into result table
    new_col = [method, "Time", "Treatment:Time", "Residual"]
    tw_rm_result.insert(0, column = (current_exp + "-dH2OvsTreatments"), value = new_col)
    tw_rm_result["F"] = tw_rm_result["F"].apply(lambda x: f"{x:.2f}")
    
    # Export to raw data to csv-file
    file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Data-Statistics/Two-Way-ANOVA-Data.csv")
    tw_rm_result.to_csv(file_name, encoding = "utf-8", index = False, header = True, mode = "a") 

IndexError: single positional indexer is out-of-bounds

# Two-Way ANOVA BM Experiment

In [ ]:
# Clean all unneeded experiment data 
twanova_df = pd.DataFrame(data_helper.filter(merged_df, time = [12]))

bm_exp = twanova_df[twanova_df["EXPERIMENT_NAME"] == "BM"]
for index in range(4, len(bm_exp.columns)):
    # TW_df MMCvsAMC
    mmc_amc = pd.DataFrame(bm_exp)
    mmc_amc.drop(mmc_amc[mmc_amc["SAMPLE_NAME"] == "CoC"].index, inplace = True)
    mmc_amc.drop(mmc_amc[mmc_amc["SAMPLE_NAME"] == "EPS"].index, inplace = True)
    # Subset data and rename current method to TREATMENT for ANOVA structure
    cur_set = mmc_amc.iloc[:, [0,1,2,3, index]]
    current_exp = cur_set["EXPERIMENT_NAME"].iloc[0]
    method = "_".join(cur_set.columns[4].split("_")[:-1]) # save current method name
    cur_set.columns = ["ID", "EXPERIMENT_NAME", "SAMPLE_NAME", "TIME", "TREATMENT"] # rename treatment column
    
    # Test normal distribution in qqplot
    plot_helper.visualize_qqplot(title = current_exp + " " + method, data = cur_set["TREATMENT"])
    # Perform two-way ANOVA
    model = ols('TREATMENT ~ C(SAMPLE_NAME) + C(TIME) + C(SAMPLE_NAME):C(TIME)', data = cur_set).fit()
    tw_bm_result = sm.stats.anova_lm(model, typ=2)
    # Insert meta information into result table
    new_col = [method, "Time", "Treatment:Time", "Residual"]
    tw_bm_result.insert(0, column = (current_exp + "-MMCvsAMC"), value = new_col)
    tw_bm_result["F"] = tw_bm_result["F"].apply(lambda x: f"{x:.2f}")
    
    # Export to raw data to csv-file
    file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Data-Statistics/Two-Way-ANOVA-Data.csv")
    tw_bm_result.to_csv(file_name, encoding = "utf-8", index = False, header = True, mode = "a")
    
    # TW_df CoCvsEPS
    coc_eps = pd.DataFrame(bm_exp)
    coc_eps.drop(coc_eps[coc_eps["SAMPLE_NAME"] == "AMC"].index, inplace = True)
    coc_eps.drop(coc_eps[coc_eps["SAMPLE_NAME"] == "CoA"].index, inplace = True)
    # Subset data and rename current method to TREATMENT for ANOVA structure
    cur_set = coc_eps.iloc[:, [0,1,2,3, index]]
    current_exp = cur_set["EXPERIMENT_NAME"].iloc[0]
    method = "_".join(cur_set.columns[4].split("_")[:-1]) # save current method name
    cur_set.columns = ["ID", "EXPERIMENT_NAME", "SAMPLE_NAME", "TIME", "TREATMENT"] # rename treatment column
    
    # Test normal distribution in qqplot
    plot_helper.visualize_qqplot(title = current_exp + " " + method, data = cur_set["TREATMENT"])
    # Perform two-way ANOVA
    model = ols('TREATMENT ~ C(SAMPLE_NAME) + C(TIME) + C(SAMPLE_NAME):C(TIME)', data = cur_set).fit()
    tw_bm_result = sm.stats.anova_lm(model, typ=2)
    # Insert meta information into result table
    new_col = [method, "Time", "Treatment:Time", "Residual"]
    tw_bm_result.insert(0, column = (current_exp + "-CoCvsEPS"), value = new_col)
    tw_bm_result["F"] = tw_bm_result["F"].apply(lambda x: f"{x:.2f}")
    
    # Export to raw data to csv-file
    file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Data-Statistics/Two-Way-ANOVA-Data.csv")
    tw_bm_result.to_csv(file_name, encoding = "utf-8", index = False, header = True, mode = "a") 

# Post-Hoc Test

In [16]:
# Function for method name check
def check_methodname(abbr_name):
    if abbr_name == "RRATE_%":
        full_name = "Recovery Rate [%]"
    elif abbr_name == "MEAN_SVELO_M/H":
        full_name = "Sinking Velocity [m h-1]"    
    elif abbr_name == "MEAN_PARSIZE_ÂµM":
        full_name = "Particle Size [µm]"
    elif abbr_name == "ZETA_POTENTIAL_MV":
        full_name = "Zeta Potential [mV]"
    else:
        full_name = abbr_name
    return full_name

def format_cell(x):
    try:
        num = float(x)
        return f"{num:.2f}"
    except:
        return f"{num:.2f}"  # Wenn es keine Zahl ist, bleibt es wie es ist

# Custom Post-Hoc for Paper

In [17]:
# Post-Hoc Test All Experiments

# Delete old file if exists
file = data_helper.get_output_path(CONFIG, "Raw-Data/Paper_Data-Statistics/PostHoc-Data.csv")
if(os.path.exists(file) and os.path.isfile(file)): 
  os.remove(file) 
  print("Old file deleted.") 
else: 
  print("File not found.")

# Get custom data frame with correct sorted samples (CAREFUL, no automated data handling here!)
posthoc_df = pd.read_csv(data_helper.get_output_path(CONFIG, "Raw-Data/Paper_Data-Statistics/custom_merged_df.csv"), index_col=0, encoding='unicode_escape')

# iterate through all unique experiments
for exp in posthoc_df["EXPERIMENT_NAME"].unique():
    sub_df = posthoc_df[posthoc_df["EXPERIMENT_NAME"] == exp]
    # For all methods
    for index in range(4, len(sub_df.columns)):
        current_sampleset = sub_df.iloc[:, [0,1,2,3, index]]
        current_exp = current_sampleset["EXPERIMENT_NAME"].iloc[0]
        method = current_sampleset.columns[4] # save current method name
        
        # For all timepoints
        for unique_time in current_sampleset["TIME"].unique(): 
            current_timeset = current_sampleset[current_sampleset["TIME"] == unique_time]
           
            # PostHoc-Tests / Table Creation
            posthoc_result = sp.posthoc_tukey(current_timeset, val_col=method, group_col='SAMPLE_NAME')
            posthoc_result = posthoc_result.map(lambda x: f"{x:.2f}" if pd.notnull(x) else "")
            
            # Sample Information Column
            sub_meta = META_DATA[META_DATA["EXP_ABBR"] == exp][:1].reset_index()
            sample_count = len(current_sampleset["SAMPLE_NAME"].unique())
            sample_information = [sub_meta["EXP_FULL"][0], check_methodname(method), str(unique_time) + " h"]
            # Match Rows of Sample Information for each Experiment and its Samples
            if sample_count > len(sample_information):
                difference = sample_count - len(sample_information)
                for filler in range(0, difference):
                    sample_information.append("")
            # Insert Information at the end
            posthoc_result.insert(len(posthoc_result), column = "Sample Information", value = sample_information)
            
            # File Export
            file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Paper_Data-Statistics/PostHoc-Data.csv")
            posthoc_result.to_csv(file_name, encoding = "utf-8-sig", index = True, header = True, mode = "a", float_format="%.2f")
            
print("New file created.") 

File not found.


FileNotFoundError: [Errno 2] No such file or directory: '/Users/cedric/Documents/13_Personal/132_Formal-Education/136_CAU-Kiel_PhD-Biology/03_Experiments/03_Output/00_Example-Output/Raw-Data/Paper_Data-Statistics/custom_merged_df.csv'

# Sample Data + ConfInt

### Confidence Intervalle für gesamten Datenframe berechnen 

In [18]:
# Beispielhafte Aggregationsfunktion, die das 95%-Konfidenzintervall zurückgibt
def ci95(x):
    ci_low, ci_high = sms.DescrStatsW(x).tconfint_mean()
    return pd.Series({'CI95': f"{ci_low:.2f}" + ", " + f"{ci_high:.2f}"})

# Deine Messwerte-Gruppierung
group_cols = ["EXPERIMENT_NAME", "SAMPLE_NAME", "TIME"]
value_cols = [
    "RRATE_%", "MEAN_SVELO_M/H", "MEAN_PARSIZE_µM", "TSS_g/L", "TSS_%",
    "VSS_g/L", "VSS_%", "ZETA_POTENTIAL_MV"
]

# Leere Liste für alle CI-DataFrames
ci_frames = []

for col in value_cols:
    # Gruppieren und CI berechnen
    ci = merged_df.groupby(group_cols)[col].apply(ci95).unstack()
    # Umbenennen der Spalten mit dem Messnamen als Prefix
    ci.columns = [f"{col}"]
    ci_frames.append(ci)

# Alles zusammenfügen
ci_df = pd.concat(ci_frames, axis=1).reset_index()

# Ausgabe: ci_df enthält alle Konfidenzintervalle pro Gruppe
print(ci_df.head())

  EXPERIMENT_NAME    SAMPLE_NAME  TIME        RRATE_% MEAN_SVELO_M/H  \
0              SC     A.obliquus  0.00    -1.38, 5.00     0.00, 0.00   
1              SC       Algaemix  0.00  91.76, 103.20     2.86, 5.46   
2              SC  C.reinhardtii  0.00   -4.16, 13.36     0.19, 0.85   
3              SC     E.texensis  0.00  -29.76, 66.90     0.77, 1.29   
4              SC  T.albertanoae  0.00   16.72, 35.42     0.56, 0.86   

   MEAN_PARSIZE_µM     TSS_g/L         TSS_%     VSS_g/L       VSS_%  \
0  332.93, 1228.79  0.96, 1.50  93.71, 96.14  0.06, 0.06  3.86, 6.29   
1  332.93, 1228.79  0.96, 1.50  93.71, 96.14  0.06, 0.06  3.86, 6.29   
2  332.93, 1228.79  0.96, 1.50  93.71, 96.14  0.06, 0.06  3.86, 6.29   
3  332.93, 1228.79  0.96, 1.50  93.71, 96.14  0.06, 0.06  3.86, 6.29   
4  332.93, 1228.79  0.96, 1.50  93.71, 96.14  0.06, 0.06  3.86, 6.29   

  ZETA_POTENTIAL_MV  
0    -20.11, -15.18  
1     -40.66, -1.45  
2    -22.64, -20.39  
3    -20.19, -17.70  
4    -30.79, -23.47  


### Merge Mean, StDev, CI into one string-based Dataframe

In [19]:
export_mean_df = mean_df
export_stdev_df = stdev_df
export_ci_df = ci_df 

# Step 3: Gemeinsame Spalten beibehalten
common_cols = ['EXPERIMENT_NAME', 'SAMPLE_NAME', 'TIME']
treatment_cols = [col for col in export_mean_df.columns if col not in common_cols]

# Step 4: Neue DataFrame vorbereiten
combined_df = export_mean_df[common_cols].copy()

# Step 5: Behandlung der Treatment-Werte im gewünschten Format (Forced 2 Nachkommastellen)
for col in treatment_cols:
    combined_df[col] = export_mean_df[col].apply(lambda x: f"{x:.2f}") + " ± " + export_stdev_df[col].apply(lambda x: f"{x:.2f}") + " [95% CI: " + export_ci_df[col] + "]"


# Export confint to csv-file (append)
file_name = data_helper.get_output_path(CONFIG, "Raw-Data/Paper_Data-Statistics/Sample-Data-With-CI95.csv")
combined_df.to_csv(file_name, encoding = "utf-8-sig", index = False, header = True, mode = "w")

combined_df.head(1)

NameError: name 'mean_df' is not defined